# Kafka Producer Test Notebook
This notebook simulates the real-time data generation for the recruitment tracking system.

In [8]:
import os
import time
import random
import datetime
import json
import pandas as pd
from dotenv import load_dotenv
from kafka import KafkaProducer
import mysql.connector
import uuid

# Load environment variables from the root directory
load_dotenv(dotenv_path="../.env")

True

In [9]:
# MySQL Configuration
MYSQL_HOST = os.getenv('MYSQL_HOST')
MYSQL_PORT = os.getenv('MYSQL_PORT')
MYSQL_DB = os.getenv('MYSQL_DB')
MYSQL_USER = os.getenv('MYSQL_USER')
MYSQL_PASSWORD = os.getenv('MYSQL_PASSWORD')

# Kafka Configuration
KAFKA_BOOTSTRAP_SERVERS = os.getenv('KAFKA_BOOTSTRAP_SERVERS')
KAFKA_TOPIC = os.getenv('KAFKA_TOPIC')

In [10]:
def get_data_from_job():
    cnx = mysql.connector.connect(
        user=MYSQL_USER, password=MYSQL_PASSWORD,
        host=MYSQL_HOST, database=MYSQL_DB
    )
    query = "SELECT id as job_id, campaign_id, group_id, company_id FROM job"
    mysql_data = pd.read_sql(query, cnx)
    cnx.close()
    return mysql_data

def get_data_from_publisher():
    cnx = mysql.connector.connect(
        user=MYSQL_USER, password=MYSQL_PASSWORD,
        host=MYSQL_HOST, database=MYSQL_DB
    )
    query = "SELECT DISTINCT(id) as publisher_id FROM master_publisher"
    mysql_data = pd.read_sql(query, cnx)
    cnx.close()
    return mysql_data

In [11]:
def json_serializer(data):
    return json.dumps(data).encode('utf-8')

# Initialize Kafka Producer
print(f"Connecting to Kafka at {KAFKA_BOOTSTRAP_SERVERS}...")
producer = KafkaProducer(
    bootstrap_servers=[KAFKA_BOOTSTRAP_SERVERS],
    value_serializer=json_serializer
)

Connecting to Kafka at broker:29092...


In [12]:
def generate_and_send_data(n_records, job_list, campaign_list, company_list, group_list, publisher_list):
    for _ in range(n_records):
        bid = str(random.randint(0, 1))
        interact = ['click', 'conversion', 'qualified', 'unqualified']
        custom_track = random.choices(interact, weights=(70, 10, 10, 10))[0]
        job_id = random.choice(job_list)
        publisher_id = random.choice(publisher_list)
        group_id = random.choice(group_list)
        campaign_id = random.choice(campaign_list)
        ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')

        record = {
            "create_time": str(uuid.uuid1()),
            "bid": bid,
            "campaign_id": int(campaign_id),
            "custom_track": custom_track,
            "group_id": int(group_id),
            "job_id": int(job_id),
            "publisher_id": int(publisher_id),
            "ts": ts
        }
        producer.send(KAFKA_TOPIC, record)
        
    producer.flush()
    print(f"--- {n_records} Records Sent to Kafka Topic '{KAFKA_TOPIC}' ---")

In [13]:
print(f"Starting Kafka Data Generator...")

# Pre-fetch dimension data
jobs_data = get_data_from_job()
publisher = get_data_from_publisher()['publisher_id'].to_list()

job_list = jobs_data['job_id'].to_list()
campaign_list = jobs_data['campaign_id'].to_list()
company_list = jobs_data['company_id'].to_list()
group_list = jobs_data[jobs_data['group_id'].notnull()]['group_id'].astype(int).to_list()

try:
    while True:
        n = random.randint(1, 20)
        generate_and_send_data(n, job_list, campaign_list, company_list, group_list, publisher)
        time.sleep(10)
except KeyboardInterrupt:
    print("\nStopping Kafka Data Generator...")
finally:
    producer.close()

Starting Kafka Data Generator...
--- 7 Records Sent to Kafka Topic 'tracking_data' ---


/tmp/ipykernel_4909/1318035422.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  mysql_data = pd.read_sql(query, cnx)
/tmp/ipykernel_4909/1318035422.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  mysql_data = pd.read_sql(query, cnx)


--- 18 Records Sent to Kafka Topic 'tracking_data' ---

Stopping Kafka Data Generator...


In [ ]:
# !rm -rf /tmp/checkpoint_kafka_to_cassandra
